In [1]:
! pip install ultralytics opencv-python scikit-learn matplotlib

In [ ]:
import os
import cv2
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
bike_dir = r"/Users/binhminh/Project-II/data/vietnam-car-license-plate/Bike/GreenParking"
bike_annotation = r"/Users/binhminh/Project-II/data/vietnam-car-license-plate/Bike/GreenParking/location.txt"
output_labels_dir = r"/Users/binhminh/Project-II/data/temp_labels" # Thư mục nháp
final_dataset = r"/Users/binhminh/Project-II/data/yolo_dataset"         # Thư mục đích chuẩn YOLO

In [ ]:
os.makedirs(output_labels_dir, exist_ok=True)
with open(bike_annotation, 'r') as f:
    lines = f.readlines()

valid_images = [] # Chỉ giữ lại các file ảnh Tồn Tại và Hợp lệ
for line in lines:
    line = line.strip()
    if not line: continue
    
    parts = line.split(' ')
    image_name = parts[0]
    class_id = 0 # 0 là class Mặc định (Biển số / License Plate)
    x_min, y_min, w_box, h_box = map(float, parts[2:6])
    
    img_path = os.path.join(bike_dir, image_name)
    if not os.path.exists(img_path): continue
        
    img = cv2.imread(img_path)
    if img is None: continue
    img_h, img_w, _ = img.shape
    
    # Tính tâm và chuẩn hoá (Normalize) về [0, 1]
    # Dataset location.txt đã CÓ SẴN toạ độ là (x_center, y_center), nên KHÔNG CỘNG THÊM w_box/2.0
    x_center = (x_min + w_box / 2.0) / img_w
    y_center = (y_min + h_box / 2.0) / img_h
    w_norm = w_box / img_w
    h_norm = h_box / img_h
    
    if w_norm > 1.0 or h_norm > 1.0: continue # Lọc dữ liệu lỗi (box to hơn ảnh)
    
    txt_name = image_name.replace('.jpg', '.txt').replace('.png', '.txt')
    txt_path = os.path.join(output_labels_dir, txt_name)
    
    with open(txt_path, 'w') as out_f:
        out_f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")
    
    valid_images.append(image_name)

print(f"Đã tạo {len(valid_images)} file labels YOLO.")


In [ ]:
# 2. Xóa Dataset cũ nếu có để tránh ghi đè rác
if os.path.exists(final_dataset):
    shutil.rmtree(final_dataset)

In [ ]:
# 3. Chia tập dữ liệu (80-10-10)
train_imgs, temp_imgs = train_test_split(valid_images, test_size=0.2, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}

for split_name, imgs in splits.items():
    img_dest = os.path.join(final_dataset, 'images', split_name)
    lbl_dest = os.path.join(final_dataset, 'labels', split_name)
    os.makedirs(img_dest, exist_ok=True)
    os.makedirs(lbl_dest, exist_ok=True)
    
    for img_name in imgs:
        lbl_name = img_name.replace('.jpg', '.txt').replace('.png', '.txt')
        # Copy file
        shutil.copy(os.path.join(bike_dir, img_name), os.path.join(img_dest, img_name))
        shutil.copy(os.path.join(output_labels_dir, lbl_name), os.path.join(lbl_dest, lbl_name))

# Xoá file nháp
shutil.rmtree(output_labels_dir)
print(f"XUYÊN SUỐT: Train({len(train_imgs)}) - Val({len(val_imgs)}) - Test({len(test_imgs)})")